In [1]:
import numpy as np
import pandas as pd

df: pd.DataFrame = pd.read_csv(
    filepath_or_buffer="data/stock prices.csv",
    usecols=["date", "close", "high", "low", "open", "adjClose", "adjHigh", "adjLow", "adjOpen"]
).dropna()

# Sorting based on date column
df = df.sort_values(by="date").set_index(keys="date", drop=True)
df.head()

,close,high,low,open,adjClose,adjHigh,adjLow,adjOpen
date,,,,,,,,
2015-05-27 00:00:00+00:00,132.045,132.260,130.05,130.34,121.682558,121.880685,119.844118,120.111360
2015-05-28 00:00:00+00:00,131.780,131.950,131.10,131.86,121.438354,121.595013,120.811718,121.512076
2015-05-29 00:00:00+00:00,130.280,131.450,129.90,131.23,120.056069,121.134251,119.705890,120.931516
2015-06-01 00:00:00+00:00,130.535,131.390,130.05,131.20,120.291057,121.078960,119.844118,120.903870
2015-06-02 00:00:00+00:00,129.960,130.655,129.32,129.86,119.761181,120.401640,119.171406,119.669029


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler


class Preprocess:
    def __init__(self, test_split: float):
        self.test_split = test_split
        self.scaler = StandardScaler()

    def splitting(self, df:pd.DataFrame):
        train_df = df.iloc[ : int(df.shape[0] * (1 - self.test_split)), : ]
        test_df = df.iloc[int(df.shape[0] * (1 - self.test_split)): , : ]
        return train_df, test_df

    def preprocess(self, train_df:pd.DataFrame, test_df:pd.DataFrame):
        train_df = self.scaler.fit_transform(train_df)
        test_df = self.scaler.transform(test_df)

        X_train = train_df[:, 1:]
        y_train= train_df[:, 1]

        X_test = test_df[:, 1:]
        y_test = test_df[:, 1]

        return X_train, X_test, y_train, y_test


class CustomDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray, timestep: int = 1):
        self.timestep = timestep

        # Dataset Preperation for stock price predictor model
        data = pd.DataFrame(data=X)
        periods = list(range(1, self.timestep + 1)) # NOTE: The current timestep is inclusive: timestep = 50 → current (1) + previous (49)
        self.X = torch.from_numpy(data.shift(periods).iloc[timestep + 1:].to_numpy()).to(torch.float32)
        self.y = torch.from_numpy(y[timestep + 1: ]).to(torch.float32)
    
    def __len__(self):
        return self.X.shape[0]
    
    def __getitem__(self, index):
        return self.X[index, :].reshape(self.timestep, self.X.shape[1] // self.timestep), self.y[index]

In [3]:
# Preprocessing and Splitting
process = Preprocess(test_split=0.2)
train_df, test_df = process.splitting(df=df)
X_train, X_test, y_train, y_test = process.preprocess(train_df, test_df)

print(
    X_train.shape, 
    X_test.shape, 
    y_train.shape, 
    y_test.shape
)

train_dataset = CustomDataset(X_train, y_train, timestep=50)
test_dataset = CustomDataset(X_test, y_test, timestep=50)

train_loader = DataLoader(dataset=train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=y_train.shape[0], shuffle=False)

print(
    train_dataset.X.shape, 
    train_dataset.y.shape, 
    test_dataset.X.shape, 
    test_dataset.y.shape
)

(1006, 7) (252, 7) (1006,) (252,)
torch.Size([955, 350]) torch.Size([955]) torch.Size([201, 350]) torch.Size([201])


In [4]:
from torch.nn import Module, RNN, Linear
from typing import Literal

device: Literal['cpu', 'cuda'] = 'cuda' if torch.cuda.is_available() else 'cpu'

class CustomModel(Module):
    def __init__(self, input_size: int):
        super().__init__()
        self.rnn = RNN(input_size=input_size, hidden_size=25, num_layers=1, nonlinearity="tanh", batch_first=True, bidirectional=False)
        self.linear = Linear(in_features=25, out_features=1)
        
    def forward(self, X_train):
        _, final = self.rnn(X_train)
        y_pred = self.linear(final.squeeze(dim=0)) # removing no. hidden states dimension
        return y_pred

In [5]:
# Training parameters
learning_rate: float = 0.001
epochs: int = 100

# Loss function and optimizer
model: CustomModel = CustomModel(input_size=train_dataset.X.shape[1] // train_dataset.timestep).to(device)
model.train()

criterion = torch.nn.modules.loss.HuberLoss(delta=1.0)
optimizer = torch.optim.Adam(params = model.parameters())

In [6]:
for _ in range(epochs):
    for feature, label in train_loader:
        feature = feature.to(device)
        label = label.to(device)
        optimizer.zero_grad()

        y_pred = model(feature)
        loss = criterion(y_pred, label.unsqueeze(dim=1))
        loss.backward()

        optimizer.step()

In [7]:
model.eval()
with torch.no_grad():
    for feature, label in test_loader:
        feature = feature.to(device)
        label = label.to(device)
        y_pred = model(feature).to('cpu')

In [15]:
import plotly.graph_objects as go

# Ensure data is 1D arrays for Plotly
# If y_test or y_pred are 2D (e.g., shape (n_samples, 1)), flatten them
actual_prices = test_dataset.y.reshape(-1) * (test_df.loc[:, 'close']).std() + (test_df.loc[:, 'close']).mean()
predicted_prices = y_pred.reshape(-1) * (test_df.loc[:, 'close']).std() + (test_df.loc[:, 'close']).mean()

# Create the x-axis (Days). 
# Assuming len(actual_prices) represents the number of days.
days = np.arange(len(actual_prices))

# Create the figure
fig = go.Figure()

# Add Actual Prices trace
fig.add_trace(go.Scatter(
    x=days,
    y=actual_prices,
    mode='lines',
    name='Actual Stock Prices',
    line=dict(color='blue', width=2)
))

# Add Predicted Prices trace
fig.add_trace(go.Scatter(
    x=days,
    y=predicted_prices,
    mode='lines',
    name='Predicted Stock Prices',
    line=dict(color='orange', width=2, dash='dash')
))

# Update layout
fig.update_layout(
    title='Stock Price Prediction',
    xaxis_title='Days',
    yaxis_title='Price',
    hovermode='x unified',
    template='plotly_white',
    width=1000,  # Adjust width to match your matplotlib figsize preference
    height=500
)

# Show the chart
fig.show()